In [1]:
from pathlib import Path
from typing import Any

import optuna
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from sklearn.preprocessing import TargetEncoder, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, KFold
from sklearn.ensemble import IsolationForest, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.dummy import DummyRegressor

/Users/cube/source/dhbw/exploration/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
Path.cwd()

PosixPath('/Users/cube/source/dhbw/exploration/src2')

In [3]:
root_dir = Path.cwd().parent
temp_dir = root_dir / ".temp"
assert temp_dir.exists()

temp_dir

PosixPath('/Users/cube/source/dhbw/exploration/.temp')

In [4]:
dataset_train_path = temp_dir / "vehicles_train.csv"
dataset_test_path = temp_dir / "vehicles_test.csv"
dataset_train_path, dataset_test_path

(PosixPath('/Users/cube/source/dhbw/exploration/.temp/vehicles_train.csv'),
 PosixPath('/Users/cube/source/dhbw/exploration/.temp/vehicles_test.csv'))

In [5]:
images_path = Path.cwd() / ".." / "charged-ieee" / "images"
images_path

PosixPath('/Users/cube/source/dhbw/exploration/src2/../charged-ieee/images')

Globals

In [6]:
RNG = 99

# Load

In [7]:
df_train, df_test = pd.read_csv(dataset_train_path), pd.read_csv(dataset_test_path)
df_train.shape, df_test.shape

((279106, 15), (93036, 15))

In [8]:
cols_iso = ["year", "odometer", "manufacturer_missing"]

# Prepare

In [9]:
col_label = "price"
cols_features = [c for c in df_train.columns if c != col_label]
col_label, len(cols_features)

('price', 14)

In [10]:
X_train, y_train = df_train[cols_features], df_train[col_label]
X_test, y_test = df_test[cols_features], df_test[col_label]
X_train.shape, y_train.shape, X_test.shape, y_test.shape

((279106, 14), (279106,), (93036, 14), (93036,))

# Train

### Common

In [11]:
def suggest_isolation_forest_params(trial: optuna.Trial) -> dict[str, Any]:
    mode = trial.suggest_categorical("iso_contamination_mode", ["auto", "manual"])

    return {
        "iso_n_estimators": trial.suggest_int("iso_n_estimators", 50, 200),
        "iso_contamination": "auto"
        if mode == "auto"
        else trial.suggest_float("iso_contamination", 0.01, 0.5),
        "iso_contamination_mode": mode,
    }

In [12]:
def new_isolation_forest(model_params, **params) -> IsolationForest:
    return IsolationForest(
        n_estimators=model_params["iso_n_estimators"],
        contamination="auto"
        if model_params["iso_contamination_mode"] == "auto"
        else model_params["iso_contamination"],
        random_state=RNG,
        **params,
    )


def fit_model_with_isolation_forest(model, X_in, y_in, model_params):
    model_iso = new_isolation_forest(model_params, n_jobs=-1)
    inlier_mask = model_iso.fit_predict(X_in[cols_iso]) == 1
    model.fit(X_in[inlier_mask], y_in[inlier_mask])
    return model

In [13]:
def new_model_pipeline(model) -> Pipeline:
    cols_cat_target = ["region", "state", "manufacturer"]
    cols_cat_one_hot = [
        "condition",
        "cylinders",
        "fuel",
        "title_status",
        "transmission",
        "drive",
        "type",
        "paint_color",
    ]
    cols_num = ["year", "odometer", "manufacturer_missing"]

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "target",
                TargetEncoder(target_type="continuous", random_state=RNG),
                cols_cat_target,
            ),
            (
                "one_hot",
                OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                cols_cat_one_hot,
            ),
            (
                "num",
                "passthrough",
                cols_num,
            ),
        ],
        sparse_threshold=0,
        verbose_feature_names_out=False,
    )

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("scaler", StandardScaler()),
            ("model", model),
        ]
    )

### Isolation Forest Parameters

In [14]:
isolation_forest_params = {
    "iso_n_estimators": 157,
    "iso_contamination_mode": "manual",
    "iso_contamination": 0.4662846821308693,
}
isolation_forest_params

{'iso_n_estimators': 157,
 'iso_contamination_mode': 'manual',
 'iso_contamination': 0.4662846821308693}

### Model Dummy (Median)

In [59]:
model_dummy = new_model_pipeline(DummyRegressor(strategy="median"))
model_dummy.fit(X_train, y_train)
y_pred_dummy = model_dummy.predict(X_test)
mae_dummy, rmse_dummy, r2_dummy = (
    mean_absolute_error(y_test, y_pred_dummy),
    root_mean_squared_error(y_test, y_pred_dummy),
    r2_score(y_test, y_pred_dummy),
)
mae_dummy, rmse_dummy, r2_dummy

(10972.87008254869, 14264.327880967614, -0.04763379647935784)

### Model Linear

In [60]:
model_linear = new_model_pipeline(LinearRegression(n_jobs=-1))
model_linear.fit(X_train, y_train)
y_pred_linear = model_linear.predict(X_test)
mae_linear, rmse_linear, r2_linear = (
    mean_absolute_error(y_test, y_pred_linear),
    root_mean_squared_error(y_test, y_pred_linear),
    r2_score(y_test, y_pred_linear),
)
mae_linear, rmse_linear, r2_linear

(6142.9573196718175, 8707.342244014852, 0.6096284996005732)

### Model RandomForest

In [16]:
def new_model_rft(model_params, **params) -> RandomForestRegressor:
    return RandomForestRegressor(
        n_estimators=model_params["rf_n_estimators"],
        max_depth=model_params["rf_max_depth"],
        min_samples_split=model_params["rf_min_samples_split"],
        min_samples_leaf=model_params["rf_min_samples_leaf"],
        random_state=RNG,
        **params,
    )

In [62]:
def objective_model_rft(trial):
    iso = new_isolation_forest(
        suggest_isolation_forest_params(trial),
        n_jobs=-1,
    )
    inlier_mask = iso.fit_predict(X_train[cols_iso]) == 1
    X_in, y_in = X_train[inlier_mask], y_train[inlier_mask]

    model_pipeline = new_model_pipeline(
        new_model_rft(
            {
                "rf_n_estimators": trial.suggest_int("rf_n_estimators", 100, 500),
                "rf_max_depth": trial.suggest_int("rf_max_depth", 4, 24),
                "rf_min_samples_split": trial.suggest_int(
                    "rf_min_samples_split", 2, 20
                ),
                "rf_min_samples_leaf": trial.suggest_int("rf_min_samples_leaf", 1, 10),
            },
            n_jobs=-1,
        )
    )

    cv = KFold(n_splits=7, shuffle=True, random_state=RNG)
    mae = -cross_val_score(
        model_pipeline, X_in, y_in, cv=cv, scoring="neg_mean_absolute_error", n_jobs=-1
    ).mean()
    return mae

In [ ]:
study_rft = optuna.create_study(direction="minimize")
study_rft.optimize(objective_model_rft, n_trials=25)
model_rft_params = study_rft.best_params
model_rft_params

[I 2026-04-20 14:43:22,625] A new study created in memory with name: no-name-31492bde-8186-4ccd-bb1f-31cd538cdb56
[I 2026-04-20 14:48:23,259] Trial 0 finished with value: 2853.7290727979207 and parameters: {'iso_contamination_mode': 'auto', 'iso_n_estimators': 165, 'rf_n_estimators': 328, 'rf_max_depth': 21, 'rf_min_samples_split': 16, 'rf_min_samples_leaf': 6}. Best is trial 0 with value: 2853.7290727979207.
[I 2026-04-20 14:53:23,715] Trial 1 finished with value: 2984.623367210032 and parameters: {'iso_contamination_mode': 'manual', 'iso_n_estimators': 102, 'iso_contamination': 0.14854935991628507, 'rf_n_estimators': 342, 'rf_max_depth': 21, 'rf_min_samples_split': 12, 'rf_min_samples_leaf': 10}. Best is trial 0 with value: 2853.7290727979207.
[I 2026-04-20 14:56:39,817] Trial 2 finished with value: 2597.3951647938075 and parameters: {'iso_contamination_mode': 'auto', 'iso_n_estimators': 144, 'rf_n_estimators': 214, 'rf_max_depth': 24, 'rf_min_samples_split': 10, 'rf_min_samples_leaf

In [17]:
model_rft_params = {
    "iso_n_estimators": 86,
    "iso_contamination_mode": "auto",
    "rf_n_estimators": 418,
    "rf_max_depth": 23,
    "rf_min_samples_split": 8,
    "rf_min_samples_leaf": 4,
}

In [18]:
model_rft = new_model_pipeline(new_model_rft(model_rft_params, n_jobs=-1))
model_rft = fit_model_with_isolation_forest(
    model_rft, X_train, y_train, model_rft_params
)
y_pred_rft = model_rft.predict(X_test)
mae_rft, rmse_rft, r2_rft = (
    mean_absolute_error(y_test, y_pred_rft),
    root_mean_squared_error(y_test, y_pred_rft),
    r2_score(y_test, y_pred_rft),
)
mae_rft, rmse_rft, r2_rft

(2985.251520593234, 5453.66379146577, 0.8468617795166383)

In [ ]:
model_rft2 = new_model_pipeline(new_model_rft(model_rft_params, n_jobs=-1))
model_rft2.fit(X_train, y_train)
y_pred_rft2 = model_rft2.predict(X_test)
mae_rft2, rmse_rft2, r2_rft2 = (
    mean_absolute_error(y_test, y_pred_rft2),
    root_mean_squared_error(y_test, y_pred_rft2),
    r2_score(y_test, y_pred_rft2),
)
mae_rft2, rmse_rft2, r2_rft2

(2533.760387278712, 4688.4508363206105, 0.8868210639923669)

### Feature Importance

In [21]:
model_rft

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('scaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('target', ...), ('one_hot', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the differe